# Métriques complètes du GAT + courbes ROC

- **Partie A (≈ 2 min, CPU)** : courbes ROC des prédictions OOF moyennées (GAT + clinique, XGBoost B4, clinique seule) → `fig_roc.pdf/png`.
- **Partie B (≈ 30–60 min, GPU recommandé)** : réentraîne les 2 GAT avec **exactement** le même protocole (CV 5×10, mêmes seeds) en gardant les prédictions de chaque fold, pour calculer AUC, AP, bACC, sensibilité et spécificité → `gat_metrics.csv`.
  Rien n'est écrasé : les nouveaux résultats vont dans `gat_results_v2.pt`.

*Exécution → Modifier le type d'exécution → GPU* (ou CPU si aucun GPU n'est disponible, ce sera juste plus long), puis *Tout exécuter*.
**Envoie-moi ensuite `gat_metrics.csv`, `roc_aucs.txt` et `fig_roc.png`.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch_geometric
import torch_geometric.typing

## Code existant (blocs 1, 2 et 3a, inchangés)

In [ ]:
# ===== BLOC 1 : modèle StrokeGAT =====
# -*- coding: utf-8 -*-
"""
================================================================================
 GAT — BLOC 1 : ARCHITECTURE  (article 2, GNN disconnectome)
================================================================================
Modele GAT end-to-end pour predire P(mRS > 2) a partir du graphe disconnectome
ChaCo, avec fusion optionnelle des variables cliniques.

Concu pour les DEUX angles de la contribution :
  - PERFORMANCE   : architecture compacte, regularisee (n=127 petit)
  - INTERPRETABILITE : chaque couche GAT peut renvoyer ses coefficients
                       d'attention (return_attention_weights) -> mapping
                       clinique ulterieur sur les regions CocoYeo143.

Concu pour la GRILLE D'ABLATION (bloc 3) via le flag use_clinical :
  - use_clinical=False -> GAT GRAPHE-SEUL   (teste l'apport de la topologie)
  - use_clinical=True  -> GAT + CLINIQUE    (modele complet)

Ossature (validee) :
  INPUT graphe (143 noeuds, x=[ChaCo, degre])
    -> GAT couche 1 (multi-tete, ELU, dropout)
    -> GAT couche 2
    -> Readout global (mean + max pooling)  = vecteur graphe
    -> [FUSION avec clinique projetee]      (si use_clinical)
    -> MLP -> logit -> sigmoide -> P(mRS>2)
--------------------------------------------------------------------------------
PREREQUIS : torch, torch_geometric (deja installes dans le notebook).
Ce bloc DEFINIT le modele ; l'entrainement est dans le bloc 2.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool


class StrokeGAT(nn.Module):
    """
    GAT 2 couches + readout mean/max + fusion clinique optionnelle + MLP.

    Parametres
    ----------
    in_channels    : dim features de noeud (ici 2 : [ChaCo regional, degre])
    hidden         : dim cachee par tete (defaut 64)
    heads          : nb de tetes d'attention couche 1 (defaut 4)
    clinical_dim   : nb de variables cliniques (ici 18) ; ignore si use_clinical=False
    clinical_proj  : dim de projection de la clinique avant fusion (defaut 16)
    dropout        : taux de dropout (defaut 0.4 ; eleve car petit dataset)
    use_clinical   : True -> fusionne la clinique ; False -> graphe seul (ablation)
    """

    def __init__(self, in_channels=2, hidden=64, heads=4,
                 clinical_dim=18, clinical_proj=16,
                 dropout=0.4, use_clinical=True):
        super().__init__()
        self.use_clinical = use_clinical
        self.dropout = dropout

        # --- Couche GAT 1 : multi-tete, sorties concatenees -> hidden*heads
        self.gat1 = GATConv(in_channels, hidden, heads=heads,
                            dropout=dropout, add_self_loops=True)
        # --- Couche GAT 2 : ramene a 'hidden', 1 tete (moyennee), pour le readout
        self.gat2 = GATConv(hidden * heads, hidden, heads=1,
                            concat=False, dropout=dropout, add_self_loops=True)

        # Readout mean+max -> vecteur graphe de dim 2*hidden
        graph_dim = hidden * 2

        # --- Projection clinique (seulement si fusion)
        if use_clinical:
            self.clin_proj = nn.Sequential(
                nn.Linear(clinical_dim, clinical_proj),
                nn.ELU(),
                nn.Dropout(dropout),
            )
            head_in = graph_dim + clinical_proj
        else:
            self.clin_proj = None
            head_in = graph_dim

        # --- Tete de prediction (MLP -> 1 logit)
        self.head = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.LayerNorm(64),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, data, return_attention=False):
        """
        data : Batch PyG (x, edge_index, batch) + data.clinical (B, clinical_dim)
        return_attention : si True, renvoie aussi les (edge_index, alpha) des 2 couches
        Retour : logits (B,) [, att1, att2]
        """
        x, edge_index, batch = data.x, data.edge_index, data.batch

        att1 = att2 = None
        if return_attention:
            x, att1 = self.gat1(x, edge_index, return_attention_weights=True)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x, att2 = self.gat2(x, edge_index, return_attention_weights=True)
            x = F.elu(x)
        else:
            x = F.elu(self.gat1(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.elu(self.gat2(x, edge_index))

        # Readout : concat mean + max pooling sur les noeuds de chaque graphe
        g = torch.cat([global_mean_pool(x, batch),
                       global_max_pool(x, batch)], dim=1)

        # Fusion clinique optionnelle
        if self.use_clinical:
            c = self.clin_proj(data.clinical)
            g = torch.cat([g, c], dim=1)

        logits = self.head(g).squeeze(-1)

        if return_attention:
            return logits, att1, att2
        return logits


# --------------------------------------------------------------------------- #
# Test rapide de coherence des dimensions (facultatif, a lancer une fois)     #
# --------------------------------------------------------------------------- #

In [ ]:
# ===== BLOC 2 : entraînement =====
# -*- coding: utf-8 -*-
"""
================================================================================
 GAT — BLOC 2 : ENTRAINEMENT + CV 5x10  (article 2, GNN disconnectome)
================================================================================
Entraine le StrokeGAT (bloc 1) selon le MEME protocole que les baselines, pour
une comparaison equitable :
   - CV stratifiee 5-fold repetee 10x (memes seeds/splits que baselines)
   - test de chaque fold INTOUCHABLE (jamais vu a l'entrainement)
   - early stopping PROPRE : split interne train/val DANS le train du fold
   - class_weight (BCEWithLogits pos_weight) pour le ratio 77/50
   - normalisation clinique z-score + imputation : fit sur TRAIN-INTERNE only
   - collecte des predictions OUT-OF-FOLD (pour tests stats du bloc 3)

Enrichit d'abord chaque graphe avec data.clinical (18 vars, memes que B4).

Sortie : dict de resultats par config + predictions poolees -> gat_results.pt
--------------------------------------------------------------------------------
PREREQUIS : bloc 1 (StrokeGAT) execute dans une cellule precedente.
"""

import os
import re
import csv
import copy
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

# ============================ A ADAPTER ========================================
MNI_DIR = "/content/drive/MyDrive/isles_prp/MNI"
PT_PATH = "/content/drive/MyDrive/isles_prp/MNI/nemo_results/pyg_dataset.pt"
OUT_PT  = "/content/drive/MyDrive/isles_prp/MNI/nemo_results/gat_results.pt"

N_SPLITS, N_REPEATS, SEED = 5, 10, 42
MAX_EPOCHS = 200
PATIENCE   = 20          # early stopping
LR         = 5e-3
WEIGHT_DECAY = 5e-4      # regularisation L2 (petit dataset)
BATCH_SIZE = 16
VAL_FRAC   = 0.2         # part du train du fold reservee a la validation interne
# ===============================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PID_RE = re.compile(r"(sub-stroke\d+)", re.IGNORECASE)

# 18 variables cliniques (identiques baseline B4, vides 100% retirees)
CLIN_NUM = ["Age", "Glucose", "Leucocytes", "CRP", "INR",
            "NIHSS at admission", "mRS premorbid"]
CLIN_BIN = ["Atrial fibrillation", "Hypertension", "Diabetes", "Hyperlipidemia",
            "Anticoagulation", "Lipid lowering drugs", "PAIs",
            "Wake-up", "In-House", "Referral"]
CLIN_COLS = CLIN_NUM + CLIN_BIN + ["Sex_M"]   # 18 colonnes


# --------------------------------------------------------------------------- #
# 1) Enrichir les graphes avec les variables cliniques (brutes, non normees)  #
# --------------------------------------------------------------------------- #
def enrich_clinical(dataset, mni_dir):
    rows = {}
    for name in os.listdir(mni_dir):
        d = os.path.join(mni_dir, name)
        if not os.path.isdir(d):
            continue
        m = PID_RE.search(name)
        if not m:
            continue
        pid = m.group(1).lower()
        f = next((os.path.join(d, fn) for fn in os.listdir(d)
                  if fn.lower().endswith("demographic_baseline.csv")), None)
        if not f:
            continue
        with open(f, newline="", encoding="utf-8-sig") as fh:
            r = csv.DictReader(fh)
            r.fieldnames = [h.strip() for h in r.fieldnames]
            row = next(r, None)
            if row:
                rows[pid] = {k.strip(): (v.strip() if v else "")
                             for k, v in row.items()}

    def tf(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return np.nan

    n_ok = 0
    for d in dataset:
        pid = d.pid.lower()
        r = rows.get(pid, {})
        vals = []
        for c in CLIN_NUM + CLIN_BIN:
            vals.append(tf(r.get(c, "")))
        sx = (r.get("Sex", "") or "").upper()
        vals.append(1.0 if sx == "M" else (0.0 if sx == "F" else np.nan))
        d.clinical_raw = torch.tensor(vals, dtype=torch.float).unsqueeze(0)  # (1,18)
        if not np.isnan(vals).all():
            n_ok += 1
    print(f"Graphes enrichis clinique : {n_ok}/{len(dataset)}")
    return dataset


# --------------------------------------------------------------------------- #
# 2) Normalisation clinique fit sur TRAIN-INTERNE (imputation + z-score)      #
# --------------------------------------------------------------------------- #
def fit_clinical_stats(dataset, idx):
    """Calcule medianes (imputation) et mean/std (z-score) sur les indices idx."""
    M = np.vstack([dataset[i].clinical_raw.numpy() for i in idx])  # (n,18)
    med = np.nanmedian(M, axis=0)
    med = np.where(np.isnan(med), 0.0, med)             # colonne 100% NaN -> 0
    Mi = np.where(np.isnan(M), med, M)
    mean = Mi.mean(axis=0)
    std = Mi.std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    return med, mean, std


def apply_clinical(dataset, med, mean, std):
    """Ecrit data.clinical (imputee + normee) sur tout le dataset."""
    for d in dataset:
        v = d.clinical_raw.numpy().copy()
        v = np.where(np.isnan(v), med, v)
        v = (v - mean) / std
        d.clinical = torch.tensor(v, dtype=torch.float)   # (1,18)


# --------------------------------------------------------------------------- #
# 3) Entrainement d'un fold avec early stopping sur validation interne        #
# --------------------------------------------------------------------------- #
def train_one_fold(dataset, tr_idx, te_idx, use_clinical, seed):
    # split interne train/val (stratifie) DANS le train du fold
    y_tr = np.array([int(dataset[i].y) for i in tr_idx])
    tin_idx, val_idx = train_test_split(
        tr_idx, test_size=VAL_FRAC, stratify=y_tr, random_state=seed)

    # stats cliniques fit sur TRAIN-INTERNE uniquement
    if use_clinical:
        med, mean, std = fit_clinical_stats(dataset, tin_idx)
        apply_clinical(dataset, med, mean, std)

    train_set = [dataset[i] for i in tin_idx]
    val_set   = [dataset[i] for i in val_idx]
    test_set  = [dataset[i] for i in te_idx]

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_set,   batch_size=64)
    test_loader  = DataLoader(test_set,  batch_size=64)

    # pos_weight pour le desequilibre (sur le train-interne)
    n1 = int(y_tr.sum()); n0 = len(y_tr) - n1
    pos_weight = torch.tensor([n0 / max(n1, 1)], dtype=torch.float, device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    torch.manual_seed(seed)
    model = StrokeGAT(use_clinical=use_clinical).to(DEVICE)   # StrokeGAT du bloc 1
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val, best_state, wait = -1.0, None, 0
    for epoch in range(MAX_EPOCHS):
        # --- train
        model.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y.float())
            loss.backward()
            opt.step()
        # --- val (AUC)
        val_auc = eval_auc(model, val_loader)
        if val_auc > best_val:
            best_val, best_state, wait = val_auc, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    # restaurer le meilleur etat, predire le TEST une seule fois
    if best_state is not None:
        model.load_state_dict(best_state)
    probs, ys = predict(model, test_loader)
    return probs, ys, te_idx


@torch.no_grad()
def eval_auc(model, loader):
    model.eval()
    ps, ys = [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        p = torch.sigmoid(model(batch)).cpu().numpy()
        ps.append(p); ys.append(batch.y.cpu().numpy())
    ps, ys = np.concatenate(ps), np.concatenate(ys)
    if len(np.unique(ys)) < 2:
        return 0.5
    return roc_auc_score(ys, ps)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    ps, ys = [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        p = torch.sigmoid(model(batch)).cpu().numpy()
        ps.append(p); ys.append(batch.y.cpu().numpy())
    return np.concatenate(ps), np.concatenate(ys)


# --------------------------------------------------------------------------- #
# 4) CV 5x10 pour une config donnee                                          #
# --------------------------------------------------------------------------- #
def run_cv(dataset, use_clinical, tag):
    y = np.array([int(d.y) for d in dataset])
    X_dummy = np.zeros((len(y), 1))
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS,
                                   random_state=SEED)
    fold_aucs = []
    # predictions out-of-fold moyennees par repetition
    oof_accum = np.zeros(len(y)); oof_count = np.zeros(len(y))

    print(f"\n>>> Config: {tag} (use_clinical={use_clinical})")
    for k, (tr, te) in enumerate(rskf.split(X_dummy, y)):
        seed_k = SEED + k
        probs, ys, te_idx = train_one_fold(dataset, tr, te, use_clinical, seed_k)
        auc = roc_auc_score(ys, probs) if len(np.unique(ys)) > 1 else 0.5
        fold_aucs.append(auc)
        oof_accum[te_idx] += probs
        oof_count[te_idx] += 1
        if (k + 1) % N_SPLITS == 0:
            rep = (k + 1) // N_SPLITS
            m = np.mean(fold_aucs[-N_SPLITS:])
            print(f"   repetition {rep:2d}/{N_REPEATS}  AUC={m:.3f}")

    fold_aucs = np.array(fold_aucs)
    oof_probs = oof_accum / np.maximum(oof_count, 1)
    print(f"   => {tag}: AUC = {fold_aucs.mean():.3f} ± {fold_aucs.std():.3f}")
    return {"tag": tag, "use_clinical": use_clinical,
            "fold_aucs": fold_aucs, "oof_probs": oof_probs, "y": y}


# --------------------------------------------------------------------------- #
# MAIN                                                                         #
# --------------------------------------------------------------------------- #
def main():
    print("=" * 70)
    print(f" GAT — CV {N_SPLITS}x{N_REPEATS} | device={DEVICE}")
    print("=" * 70)

    dataset = torch.load(PT_PATH, weights_only=False)
    print(f"Graphes charges : {len(dataset)}")
    dataset = enrich_clinical(dataset, MNI_DIR)

    results = {}
    # Deux bras cle pour l'ablation (bloc 3 ajoutera les comparaisons stats)
    results["GAT_graphe_seul"] = run_cv(dataset, use_clinical=False,
                                        tag="GAT graphe-seul")
    results["GAT_clinique"]    = run_cv(dataset, use_clinical=True,
                                        tag="GAT + clinique")

    torch.save(results, OUT_PT)
    print(f"\nResultats sauvegardes : {OUT_PT}")
    print("\n--- RESUME ---")
    for k, r in results.items():
        a = r["fold_aucs"]
        print(f"  {r['tag']:<22} AUC = {a.mean():.3f} ± {a.std():.3f}")
    print("\n  (rappel baseline B4 ChaCo+Clin XGB = 0.836)")

In [ ]:
# ===== BLOC 3a : baselines OOF + DeLong =====
# -*- coding: utf-8 -*-
"""
================================================================================
 GAT — BLOC 3a : TESTS STATISTIQUES (DeLong)  (article 2, GNN disconnectome)
================================================================================
Compare les AUC out-of-fold (OOF) des modeles DEUX A DEUX sur les MEMES patients
et les MEMES splits, via le test de DeLong (comparaison de deux courbes ROC
correlees). Repond a la question du jury : "la difference est-elle significative
ou dans le bruit ?".

Comparaisons cle :
   C1. GAT graphe-seul   vs  ChaCo-vecteur (XGB)   -> la topologie apporte-t-elle qqch ?
   C2. GAT + clinique     vs  ChaCo+Clin (XGB=B4)   -> le GAT complet est-il inferieur ?
   C3. GAT + clinique     vs  Clinique seule        -> le disconnectome ajoute-t-il ?

Principe :
   - on recharge les predictions OOF du GAT (gat_results.pt, bloc 2)
   - on regenere les predictions OOF des baselines sur les MEMES splits (SEED=42)
   - DeLong donne p-value + IC de la difference d'AUC
--------------------------------------------------------------------------------
PREREQUIS : numpy, scipy, sklearn, xgboost (deja installes). gat_results.pt present.
"""

import os
import re
import csv
import numpy as np
import pandas as pd
import torch
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# ============================ A ADAPTER ========================================
MNI_DIR = "/content/drive/MyDrive/isles_prp/MNI"
PT_PATH = "/content/drive/MyDrive/isles_prp/MNI/nemo_results/pyg_dataset.pt"
GAT_PT  = "/content/drive/MyDrive/isles_prp/MNI/nemo_results/gat_results.pt"
OUT_CSV = "/content/drive/MyDrive/isles_prp/MNI/nemo_results/delong_tests.csv"

N_SPLITS, N_REPEATS, SEED = 5, 10, 42
MRS_THRESHOLD = 2
# ===============================================================================

PID_RE = re.compile(r"(sub-stroke\d+)", re.IGNORECASE)
CLIN_NUM = ["Age", "Glucose", "Leucocytes", "CRP", "INR",
            "NIHSS at admission", "mRS premorbid"]
CLIN_BIN = ["Atrial fibrillation", "Hypertension", "Diabetes", "Hyperlipidemia",
            "Anticoagulation", "Lipid lowering drugs", "PAIs",
            "Wake-up", "In-House", "Referral"]


# --------------------------------------------------------------------------- #
# Implementation DeLong (comparaison de 2 AUC correlees, meme echantillon)     #
#   Retourne : auc1, auc2, difference, z, p-value (bilaterale), IC95 de diff   #
#   Base sur DeLong et al. 1988 (algo rapide de Sun & Xu 2014).                #
# --------------------------------------------------------------------------- #
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_transposed, label_1_count):
    m = label_1_count
    n = preds_sorted_transposed.shape[1] - m
    positive = preds_sorted_transposed[:, :m]
    negative = preds_sorted_transposed[:, m:]
    k = preds_sorted_transposed.shape[0]
    tx = np.empty((k, m), dtype=float)
    ty = np.empty((k, n), dtype=float)
    tz = np.empty((k, m + n), dtype=float)
    for r in range(k):
        tx[r] = _compute_midrank(positive[r])
        ty[r] = _compute_midrank(negative[r])
        tz[r] = _compute_midrank(preds_sorted_transposed[r])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def delong_test(y_true, score1, score2):
    """Compare deux scores sur le meme y_true. Retourne dict."""
    y_true = np.asarray(y_true)
    order = (-y_true).argsort(kind="mergesort")  # positifs d'abord
    label_1_count = int(y_true.sum())
    preds = np.vstack((score1, score2))[:, order]
    aucs, cov = _fast_delong(preds, label_1_count)
    auc1, auc2 = float(aucs[0]), float(aucs[1])
    diff = auc1 - auc2
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    se = np.sqrt(var) if var > 0 else 1e-12
    z = diff / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    ci = (diff - 1.96 * se, diff + 1.96 * se)
    return {"auc1": auc1, "auc2": auc2, "diff": diff,
            "z": float(z), "p": float(p), "ci": ci}


# --------------------------------------------------------------------------- #
# Chargement des donnees (alignees sur l'ordre du dataset PyG)                 #
# --------------------------------------------------------------------------- #
def load_data():
    ds = torch.load(PT_PATH, weights_only=False)
    pids = [d.pid.lower() for d in ds]
    y = np.array([int(d.y) for d in ds])
    X_chaco = np.vstack([d.x[:, 0].numpy() for d in ds])  # (n,143)

    rows = {}
    for name in os.listdir(MNI_DIR):
        d = os.path.join(MNI_DIR, name)
        if not os.path.isdir(d):
            continue
        m = PID_RE.search(name)
        if not m:
            continue
        pid = m.group(1).lower()
        f = next((os.path.join(d, fn) for fn in os.listdir(d)
                  if fn.lower().endswith("demographic_baseline.csv")), None)
        if not f:
            continue
        with open(f, newline="", encoding="utf-8-sig") as fh:
            r = csv.DictReader(fh)
            r.fieldnames = [h.strip() for h in r.fieldnames]
            row = next(r, None)
            if row:
                rows[pid] = {k.strip(): (v.strip() if v else "")
                             for k, v in row.items()}

    def tf(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return np.nan

    clin = []
    for pid in pids:
        rr = rows.get(pid, {})
        feat = [tf(rr.get(c, "")) for c in CLIN_NUM + CLIN_BIN]
        sx = (rr.get("Sex", "") or "").upper()
        feat.append(1.0 if sx == "M" else (0.0 if sx == "F" else np.nan))
        clin.append(feat)
    X_clin = np.array(clin)
    return pids, y, X_chaco, X_clin


def make_logreg():
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000,
                                                class_weight="balanced"))])


def make_xgb():
    clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        eval_metric="logloss", random_state=SEED)
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("clf", clf)])


def oof_predictions(X, y, model_fn):
    """Predictions out-of-fold moyennees sur les repetitions, memes splits."""
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS,
                                   random_state=SEED)
    accum = np.zeros(len(y)); count = np.zeros(len(y))
    for tr, te in rskf.split(X, y):
        model = model_fn()
        model.fit(X[tr], y[tr])
        p = model.predict_proba(X[te])[:, 1]
        accum[te] += p; count[te] += 1
    return accum / np.maximum(count, 1)


# --------------------------------------------------------------------------- #
# MAIN                                                                         #
# --------------------------------------------------------------------------- #

## Partie A — Courbes ROC

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
FIG = "/content/drive/MyDrive/isles_prp/MNI/nemo_results/figures_article"; os.makedirs(FIG, exist_ok=True)
gat = torch.load(GAT_PT, weights_only=False)
pids, y, X_chaco, X_clin = load_data()
assert np.array_equal(y, gat["GAT_clinique"]["y"]), "ordre des patients différent !"
oof = {"GAT + clinical": gat["GAT_clinique"]["oof_probs"],
       "XGBoost ChaCo + clinical (B4)": oof_predictions(np.hstack([X_chaco, X_clin]), y, make_xgb),
       "Clinical only, LogReg (B1)": oof_predictions(X_clin, y, make_logreg)}
cols = {"GAT + clinical": "#2b59c3", "XGBoost ChaCo + clinical (B4)": "#555555", "Clinical only, LogReg (B1)": "#d95f02"}
lines = []
fig, ax = plt.subplots(figsize=(4.2, 4.0))
for k, p in oof.items():
    fpr, tpr, _ = roc_curve(y, p); a = roc_auc_score(y, p); lines.append(f"{k}: AUC = {a:.3f}")
    ax.plot(fpr, tpr, lw=1.8, color=cols[k], label=f"{k} (AUC = {a:.2f})")
ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=0.8)
ax.set_xlabel("1 − specificity"); ax.set_ylabel("Sensitivity"); ax.set_aspect("equal")
ax.legend(loc="lower right", fontsize=7, frameon=False); ax.spines[["top", "right"]].set_visible(False)
for e in ("pdf", "png"): fig.savefig(f"{FIG}/fig_roc.{e}", dpi=300, bbox_inches="tight")
open(f"{FIG}/roc_aucs.txt", "w").write("\n".join(lines)); print("\n".join(lines)); plt.show()

## Partie B — Métriques complètes du GAT (réentraînement)

In [ ]:
from sklearn.metrics import average_precision_score, balanced_accuracy_score, recall_score
import pandas as pd

def run_cv_full(dataset, use_clinical, tag):
    """Identique à run_cv du bloc 2, mais garde les métriques de chaque fold."""
    y = np.array([int(d.y) for d in dataset])
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    M = {"AUC": [], "AP": [], "bACC": [], "Sens": [], "Spec": []}
    acc = np.zeros(len(y)); cnt = np.zeros(len(y))
    print(f"\n>>> {tag}")
    for k, (tr, te) in enumerate(rskf.split(np.zeros((len(y), 1)), y)):
        probs, ys, te_idx = train_one_fold(dataset, tr, te, use_clinical, SEED + k)
        pred = (probs >= 0.5).astype(int)
        M["AUC"].append(roc_auc_score(ys, probs)); M["AP"].append(average_precision_score(ys, probs))
        M["bACC"].append(balanced_accuracy_score(ys, pred))
        M["Sens"].append(recall_score(ys, pred, pos_label=1)); M["Spec"].append(recall_score(ys, pred, pos_label=0))
        acc[te_idx] += probs; cnt[te_idx] += 1
        if (k + 1) % N_SPLITS == 0: print(f"   repetition {(k+1)//N_SPLITS}/{N_REPEATS}  AUC={np.mean(M['AUC'][-N_SPLITS:]):.3f}")
    return {k: np.array(v) for k, v in M.items()} | {"oof_probs": acc / np.maximum(cnt, 1), "y": y, "tag": tag}

dataset = enrich_clinical(torch.load(PT_PATH, weights_only=False), MNI_DIR)
res = {"GAT_graphe_seul": run_cv_full(dataset, False, "GAT graph-only"),
       "GAT_clinique":    run_cv_full(dataset, True,  "GAT + clinical")}
torch.save(res, GAT_PT.replace(".pt", "_v2.pt"))
rows = []
for r in res.values():
    rows.append({"Model": r["tag"], "AUC": f"{r['AUC'].mean():.3f} ± {r['AUC'].std():.3f}",
                 **{m: f"{r[m].mean():.3f}" for m in ["AP", "bACC", "Sens", "Spec"]},
                 "AUC_OOF_averaged": f"{roc_auc_score(r['y'], r['oof_probs']):.3f}"})
T = pd.DataFrame(rows); T.to_csv(f"{FIG}/gat_metrics.csv", index=False); print(T.to_string(index=False))
print("\n(rappel version d'origine : graph-only 0.657 ± 0.098 ; + clinical 0.789 ± 0.093)")